In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("C:\\Home Credit\\application_train.csv")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 307511
Columns: 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
null_percent = df.isna().mean() * 100
columns_to_drop = null_percent[null_percent > 30].index

df.drop(columns=columns_to_drop, inplace=True)

for column in df.columns:
    if df[column].isna().sum():
        if pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].fillna(df[column].median())
        else:
            mode = df[column].mode(dropna=True)
            if not mode.empty:
                df[column] = df[column].fillna(mode.iloc[0])

print(f"Dropped {len(columns_to_drop)} columns")
print(f"Remaining missing values: {df.isna().sum().sum()}")

Dropped 50 columns
Remaining missing values: 0


In [5]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoders = {}

categorical_columns = df_encoded.select_dtypes(include=["object", "category"]).columns

for column in categorical_columns:
    label_encoders[column] = LabelEncoder()
    df_encoded[column] = label_encoders[column].fit_transform(df_encoded[column].astype(str))

print(f"Encoded {len(categorical_columns)} categorical columns:")
print(list(categorical_columns))
df_encoded.head()

Encoded 11 categorical columns:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,0,1,0,1,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,0,0,0,0,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,1,1,1,1,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,0,0,0,1,0,135000.0,312682.5,29686.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
4,100007,0,0,1,0,1,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df['TARGET'].value_counts(normalize=True)*100

TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["SK_ID_CURR", "TARGET"])
y = df_encoded["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Training features after SMOTE:", X_train_smote.shape)
print("Training labels after SMOTE:")
print(y_train_smote.value_counts())

Training features after SMOTE: (452296, 70)
Training labels after SMOTE:
TARGET
0    226148
1    226148
Name: count, dtype: int64


In [8]:
print("X_train:")
print(X_train)
print("\nX_test:")
print(X_test)
print("\ny_train:")
print(y_train)
print("\ny_test:")
print(y_test)

X_train:
        NAME_CONTRACT_TYPE  CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY  \
181648                   0            0             0                0   
229245                   0            1             1                1   
122525                   0            1             0                1   
306311                   0            1             0                0   
300658                   0            0             0                1   
...                    ...          ...           ...              ...   
31304                    1            0             0                1   
121193                   0            0             0                0   
248504                   0            0             0                0   
175469                   0            0             0                1   
285162                   0            0             1                1   

        CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
181648             2           9000

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier

models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(
    ),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier()
    )
}

accuracies = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracies[name] = accuracy_score(y_test, y_pred)

for name, score in accuracies.items():
    print(f"{name} Accuracy: {score:.4f}")

Decision Tree Accuracy: 0.8501
Random Forest Accuracy: 0.9194
Bagging Accuracy: 0.9162


In [10]:
def add_engineered_features(data):
    features = data.copy()

    features["CREDIT_TO_INCOME"] = (
        features["AMT_CREDIT"] / features["AMT_INCOME_TOTAL"].replace(0, float("nan"))
    ).fillna(0)
    features["ANNUITY_TO_INCOME"] = (
        features["AMT_ANNUITY"] / features["AMT_INCOME_TOTAL"].replace(0, float("nan"))
    ).fillna(0)
    features["CREDIT_TO_ANNUITY"] = (
        features["AMT_CREDIT"] / features["AMT_ANNUITY"].replace(0, float("nan"))
    ).fillna(0)
    features["GOODS_TO_CREDIT"] = (
        features["AMT_GOODS_PRICE"] / features["AMT_CREDIT"].replace(0, float("nan"))
    ).fillna(0)
    features["AGE_YEARS"] = features["DAYS_BIRTH"].abs() / 365.25
    features["EMPLOYED_YEARS"] = (
        features["DAYS_EMPLOYED"].where(features["DAYS_EMPLOYED"] < 0, 0).abs() / 365.25
    )
    features["EMPLOYMENT_DAYS_ANOMALY"] = (
        features["DAYS_EMPLOYED"] > 0
    ).astype(int)

    return features


X_train_fe = add_engineered_features(X_train_smote)
X_test_fe = add_engineered_features(X_test)

feature_models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
    ),
}

feature_accuracies = {}

for model_name, classifier in feature_models.items():
    classifier.fit(X_train_fe, y_train_smote)
    predictions = classifier.predict(X_test_fe)
    feature_accuracies[model_name] = accuracy_score(y_test, predictions)
    print(f"{model_name} accuracy: {feature_accuracies[model_name]:.4f}")

Decision Tree accuracy: 0.8291
Random Forest accuracy: 0.9125
Bagging accuracy: 0.9108


In [11]:
from sklearn.model_selection import GridSearchCV

models_to_tune = {
    "Decision Tree": (
        DecisionTreeClassifier(),
        {
            "max_depth": [10, None],
            "min_samples_leaf": [1, 5],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(),
        {
            "n_estimators": [100],
            "max_depth": [15, None],
        },
    ),
    "Bagging": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            random_state=42,
            n_jobs=1,
        ),
        {
            "n_estimators": [25, 50],
            "max_samples": [0.8],
        },
    ),
}

grid_results = {}

for model_name, (estimator, param_grid) in models_to_tune.items():
    cv = 5
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train_smote, y_train_smote)

    test_predictions = search.best_estimator_.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_predictions)

    grid_results[model_name] = {
        "best_params": search.best_params_,
        "cv_accuracy": search.best_score_,
        "test_accuracy": test_accuracy,
    }

    print(
        f"{model_name}: CV accuracy={search.best_score_:.4f}, "
        f"test accuracy={test_accuracy:.4f}, "
        f"best parameters={search.best_params_}"
    )

Decision Tree: CV accuracy=0.8950, test accuracy=0.8511, best parameters={'max_depth': None, 'min_samples_leaf': 5}
Random Forest: CV accuracy=0.9476, test accuracy=0.9136, best parameters={'max_depth': None, 'n_estimators': 100}
Bagging: CV accuracy=0.9441, test accuracy=0.9157, best parameters={'max_samples': 0.8, 'n_estimators': 50}


In [ ]:
import pickle
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

model_candidates = {
    "Decision Tree": (
        DecisionTreeClassifier(),
        X_train,
        y_train,
        X_test,
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        X_train,
        y_train,
        X_test,
    ),
    "Bagging": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            random_state=42,
        ),
        X_train,
        y_train,
        X_test,
    ),
    "Decision Tree (Engineered)": (
        DecisionTreeClassifier(),
        X_train_fe,
        y_train_smote,
        X_test_fe,
    ),
    "Random Forest (Engineered)": (
        RandomForestClassifier(random_state=42),
        X_train_fe,
        y_train_smote,
        X_test_fe,
    ),
    "Bagging (Engineered)": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            random_state=42,
        ),
        X_train_fe,
        y_train_smote,
        X_test_fe,
    ),
}

tuned_models = {
    "Decision Tree (Tuned)": (
        DecisionTreeClassifier(),
        {
            "max_depth": [10, None],
            "min_samples_leaf": [1, 5],
        },
        X_train_smote,
        y_train_smote,
        X_test,
    ),
    "Random Forest (Tuned)": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [100],
            "max_depth": [15, None],
        },
        X_train_smote,
        y_train_smote,
        X_test,
    ),
    "Bagging (Tuned)": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            random_state=42,
            n_jobs=1,
        ),
        {
            "n_estimators": [25, 50],
            "max_samples": [0.8],
        },
        X_train_smote,
        y_train_smote,
        X_test,
    ),
}

results = {}

for name, (model, train_X, train_y, test_X) in model_candidates.items():
    model.fit(train_X, train_y)
    predictions = model.predict(test_X)
    accuracy = accuracy_score(y_test, predictions)

    results[name] = {
        "model": model,
        "accuracy": accuracy,
        "data": test_X,
    }

for name, (model, param_grid, train_X, train_y, test_X) in tuned_models.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        refit=True,
    )
    search.fit(train_X, train_y)

    predictions = search.best_estimator_.predict(test_X)
    accuracy = accuracy_score(y_test, predictions)

    results[name] = {
        "model": search.best_estimator_,
        "accuracy": accuracy,
        "data": test_X,
    }

best_name, best_result = max(results.items(), key=lambda item: item[1]["accuracy"])

print(f"Best model: {best_name}")
print(f"Test accuracy: {best_result['accuracy']:.4f}")

pickle_path = "best_model.pickle"
with open(pickle_path, "wb") as file:
    pickle.dump(best_result["model"], file)

print(f"Saved best model to '{pickle_path}'")

with open(pickle_path, "rb") as file:
    loaded_model = pickle.load(file)

test_predictions = loaded_model.predict(best_result["data"])

Best model: Random Forest
Test accuracy: 0.9193
Saved best model to 'best_model.pickle'
